In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================================
# LOAD RAW DATA
# ============================================================

df = pd.read_pickle('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_raw.pkl')

print("="*80)
print("FEATURE ENGINEERING — NAMCS HC 2024")
print("="*80)
print(f"Starting with {len(df):,} records\n")

# ============================================================
# HANDLE MISSING VALUE CODES
# ============================================================

print("STEP 1: Converting missing value codes to NaN")
print("-"*80)

# NAMCS uses -9, -8, -7 for different types of missing data
missing_codes = [-9.0, -8.0, -7.0, -6.0]

# Apply to numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col] = df[col].replace(missing_codes, np.nan)

# Apply to diagnosis string columns (they have '-9' as string)
dx_cols = [col for col in df.columns if col.startswith('DX') and not col.startswith('DX_TYPE')]
for col in dx_cols:
    df[col] = df[col].replace(['-9', '-8', '-7', '-6'], np.nan)

print(f"✓ Converted missing codes to NaN for {len(numeric_cols)} numeric columns")
print(f"✓ Converted missing codes to NaN for {len(dx_cols)} diagnosis columns")

# ============================================================
# CHRONIC CONDITION IDENTIFICATION
# ============================================================

print("\nSTEP 2: Identifying Chronic Conditions from ICD-10-CM Codes")
print("-"*80)

# Define ICD-10-CM code ranges for chronic conditions
chronic_conditions = {
    'diabetes': {
        'codes': ['E08', 'E09', 'E10', 'E11', 'E13'],
        'description': 'Diabetes mellitus (all types)'
    },
    'hypertension': {
        'codes': ['I10', 'I11', 'I12', 'I13', 'I15'],
        'description': 'Hypertensive diseases'
    },
    'mental_health': {
        'codes': ['F20', 'F21', 'F22', 'F23', 'F24', 'F25',  # Schizophrenia spectrum
                  'F30', 'F31', 'F32', 'F33', 'F34', 'F39',  # Mood disorders
                  'F40', 'F41', 'F42', 'F43', 'F44', 'F45', 'F48'],  # Anxiety disorders
        'description': 'Mental, behavioral disorders'
    },
    'obesity': {
        'codes': ['E66'],
        'description': 'Overweight and obesity'
    },
    'asthma_copd': {
        'codes': ['J44', 'J45', 'J46'],
        'description': 'Asthma and COPD'
    },
    'chd': {
        'codes': ['I20', 'I21', 'I22', 'I23', 'I24', 'I25'],
        'description': 'Ischemic heart disease'
    },
    'ckd': {
        'codes': ['N18'],
        'description': 'Chronic kidney disease'
    },
    'substance_use': {
        'codes': ['F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19'],
        'description': 'Substance use disorders'
    }
}

# Create binary flags for each condition (any-listed diagnosis)
for condition, info in chronic_conditions.items():
    codes = info['codes']
    
    # Check all DX1-DX30 columns
    df[condition] = 0
    for dx_col in dx_cols:
        # For each diagnosis column, check if it starts with any of the condition codes
        for code in codes:
            mask = df[dx_col].notna() & df[dx_col].str.startswith(code)
            df.loc[mask, condition] = 1
    
    count = df[condition].sum()
    pct = (count / len(df)) * 100
    print(f"  {condition:20s}: {count:7,} visits ({pct:5.2f}%) - {info['description']}")

# Create composite chronic disease flag (any chronic condition)
chronic_cols = list(chronic_conditions.keys())
df['any_chronic'] = df[chronic_cols].max(axis=1)

print(f"\n  {'any_chronic':20s}: {df['any_chronic'].sum():7,} visits ({df['any_chronic'].mean()*100:5.2f}%)")

# ============================================================
# DEMOGRAPHIC FEATURE ENGINEERING
# ============================================================

print("\nSTEP 3: Demographic Feature Engineering")
print("-"*80)

# Age groups (more granular for health analysis)
df['age_group_detailed'] = pd.cut(
    df['AGE'],
    bins=[0, 5, 12, 18, 35, 50, 65, 75, 90],
    labels=['0-5', '6-12', '13-18', '19-35', '36-50', '51-65', '66-75', '75+'],
    include_lowest=True
)

print(f"Age groups created:")
print(df['age_group_detailed'].value_counts().sort_index())

# Sex (convert to binary, handle missing)
df['sex_binary'] = df['SEX'].map({1.0: 0, 2.0: 1})  # 0=Male, 1=Female
print(f"\nSex distribution:")
print(f"  Male: {(df['sex_binary']==0).sum():,}")
print(f"  Female: {(df['sex_binary']==1).sum():,}")
print(f"  Missing: {df['sex_binary'].isnull().sum():,}")

# Race/Ethnicity (create simplified categories)
df['race_simple'] = df['RACERETH'].map({
    1.0: 'White',
    2.0: 'Black',
    3.0: 'Hispanic',
    4.0: 'Other'
})

print(f"\nRace/Ethnicity distribution:")
print(df['race_simple'].value_counts(dropna=False))

# ============================================================
# VISIT COMPLEXITY METRICS
# ============================================================

print("\nSTEP 4: Visit Complexity Metrics")
print("-"*80)

# Count number of diagnoses per visit
df['num_diagnoses'] = df[dx_cols].notna().sum(axis=1)

print(f"Diagnoses per visit:")
print(df['num_diagnoses'].describe())

# High complexity visit flag (3+ diagnoses)
df['high_complexity'] = (df['num_diagnoses'] >= 3).astype(int)

print(f"\nHigh complexity visits (3+ diagnoses): {df['high_complexity'].sum():,} ({df['high_complexity'].mean()*100:.1f}%)")

# Multimorbidity (2+ chronic conditions)
df['num_chronic'] = df[chronic_cols].sum(axis=1)
df['multimorbidity'] = (df['num_chronic'] >= 2).astype(int)

print(f"Multimorbidity (2+ chronic conditions): {df['multimorbidity'].sum():,} ({df['multimorbidity'].mean()*100:.1f}%)")

# ============================================================
# TEMPORAL FEATURES
# ============================================================

print("\nSTEP 5: Temporal Features")
print("-"*80)

# Season
df['season'] = df['MONTH'].map({
    12.0: 'Winter', 1.0: 'Winter', 2.0: 'Winter',
    3.0: 'Spring', 4.0: 'Spring', 5.0: 'Spring',
    6.0: 'Summer', 7.0: 'Summer', 8.0: 'Summer',
    9.0: 'Fall', 10.0: 'Fall', 11.0: 'Fall'
})

print(f"Seasonal distribution:")
print(df['season'].value_counts())

# Weekend vs weekday
df['is_weekend'] = df['DAY'].isin([1.0, 7.0]).astype(int)  # 1=Sunday, 7=Saturday

print(f"\nWeekend vs Weekday:")
print(f"  Weekday: {(df['is_weekend']==0).sum():,} ({(df['is_weekend']==0).mean()*100:.1f}%)")
print(f"  Weekend: {(df['is_weekend']==1).sum():,} ({(df['is_weekend']==1).mean()*100:.1f}%)")

# ============================================================
# PREVENTIVE CARE FLAGS
# ============================================================

print("\nSTEP 6: Preventive Care Visit Identification")
print("-"*80)

# Well visit / preventive care codes
preventive_codes = ['Z00', 'Z01', 'Z02', 'Z76']  # General exam, screening, administrative

df['preventive_visit'] = 0
for dx_col in dx_cols:
    for code in preventive_codes:
        mask = df[dx_col].notna() & df[dx_col].str.startswith(code)
        df.loc[mask, 'preventive_visit'] = 1

print(f"Preventive care visits: {df['preventive_visit'].sum():,} ({df['preventive_visit'].mean()*100:.1f}%)")

# Vaccine administration codes
vaccine_codes = ['Z23']
df['vaccine_visit'] = 0
for dx_col in dx_cols:
    mask = df[dx_col].notna() & df[dx_col].str.startswith('Z23')
    df.loc[mask, 'vaccine_visit'] = 1

print(f"Vaccine visits: {df['vaccine_visit'].sum():,} ({df['vaccine_visit'].mean()*100:.1f}%)")

# ============================================================
# SAVE ENGINEERED FEATURES
# ============================================================

print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)

# List of all engineered features
engineered_features = (
    chronic_cols + 
    ['any_chronic', 'num_chronic', 'multimorbidity',
     'age_group_detailed', 'sex_binary', 'race_simple',
     'num_diagnoses', 'high_complexity',
     'season', 'is_weekend',
     'preventive_visit', 'vaccine_visit']
)

print(f"\nCreated {len(engineered_features)} new features:")
for feat in engineered_features:
    print(f"  - {feat}")

# Save processed dataset
output_path = '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_processed.pkl'
df.to_pickle(output_path)
print(f"\n✓ Saved processed data: {output_path}")

# Also save as CSV for easier inspection
csv_path = '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_processed.csv'
df.to_csv(csv_path, index=False)
print(f"✓ Saved processed data: {csv_path}")

# Create feature summary
print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"Total records: {len(df):,}")
print(f"Total variables: {len(df.columns)}")
print(f"\nKey metrics:")
print(f"  - Visits with any chronic condition: {df['any_chronic'].sum():,} ({df['any_chronic'].mean()*100:.1f}%)")
print(f"  - High complexity visits: {df['high_complexity'].sum():,} ({df['high_complexity'].mean()*100:.1f}%)")
print(f"  - Multimorbidity: {df['multimorbidity'].sum():,} ({df['multimorbidity'].mean()*100:.1f}%)")
print(f"  - Average diagnoses per visit: {df['num_diagnoses'].mean():.2f}")

print("\n✓ Ready for exploratory data analysis and modeling")

FEATURE ENGINEERING — NAMCS HC 2024
Starting with 503,799 records

STEP 1: Converting missing value codes to NaN
--------------------------------------------------------------------------------
✓ Converted missing codes to NaN for 44 numeric columns
✓ Converted missing codes to NaN for 30 diagnosis columns

STEP 2: Identifying Chronic Conditions from ICD-10-CM Codes
--------------------------------------------------------------------------------
  diabetes            :  52,187 visits (10.36%) - Diabetes mellitus (all types)
  hypertension        :  77,716 visits (15.43%) - Hypertensive diseases
  mental_health       :  92,453 visits (18.35%) - Mental, behavioral disorders
  obesity             :  63,477 visits (12.60%) - Overweight and obesity
  asthma_copd         :  27,178 visits ( 5.39%) - Asthma and COPD
  chd                 :   7,750 visits ( 1.54%) - Ischemic heart disease
  ckd                 :   9,029 visits ( 1.79%) - Chronic kidney disease
  substance_use       :  34,441 vi